In [8]:
!pip install flask pycryptodome pyngrok

from flask import Flask, request, render_template_string
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import base64
from pyngrok import ngrok
app = Flask(__name__)

def caesar_encrypt(text, shift=3):
    result = ""
    for char in text:
        if char.isalpha():
            shift_base = 65 if char.isupper() else 97
            result += chr((ord(char) - shift_base + shift) % 26 + shift_base)
        else:
            result += char
    return result

def aes_encrypt(text):
    key = get_random_bytes(16)
    cipher = AES.new(key, AES.MODE_EAX)
    ciphertext, tag = cipher.encrypt_and_digest(text.encode())

    return base64.b64encode(key).decode(), base64.b64encode(cipher.nonce).decode(), base64.b64encode(ciphertext).decode()

def base64_encrypt(text):
    return base64.b64encode(text.encode()).decode()

HTML_PAGE = '''
<!DOCTYPE html>
<html>
<head>
    <title>Encryption Tool</title>
    <style>
        * {
            box-sizing: border-box;
            font-family: 'Segoe UI', sans-serif;
        }

        body {
            margin: 0;
            height: 100vh;
            background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
            display: flex;
            align-items: center;
            justify-content: center;
            color: white;
        }

        .container {
            width: 500px;
            padding: 30px;
            border-radius: 20px;
            background: rgba(255,255,255,0.05);
            backdrop-filter: blur(20px);
            box-shadow: 0 0 40px rgba(0,0,0,0.5);
        }

        h1 {
            text-align: center;
            margin-bottom: 20px;
            font-weight: 600;
        }

        textarea {
            width: 100%;
            height: 120px;
            padding: 12px;
            border-radius: 10px;
            border: none;
            outline: none;
            margin-bottom: 15px;
            resize: none;
        }

        select {
            width: 100%;
            padding: 10px;
            border-radius: 10px;
            border: none;
            margin-bottom: 15px;
        }

        button {
            width: 100%;
            padding: 12px;
            border: none;
            border-radius: 10px;
            background: #00c9ff;
            color: black;
            font-weight: bold;
            cursor: pointer;
            transition: 0.3s;
        }

        button:hover {
            background: #92fe9d;
        }

        .output {
            margin-top: 20px;
        }

        .copy-btn {
            margin-top: 10px;
            background: #ffffff;
            color: black;
        }

        .error {
            color: #ff6b6b;
            text-align: center;
        }
    </style>
</head>
<body>

<div class="container">
    <h1>Text Encryption Tool</h1>

    <form method="POST">
        <textarea name="plaintext" placeholder="Enter your text here"></textarea>

        <select name="method">
            <option value="">Select Encryption</option>
            <option value="caesar">Caesar Cipher</option>
            <option value="aes">AES</option>
            <option value="base64">Base64</option>
        </select>

        <button type="submit">Encrypt</button>
    </form>

    {% if error %}
        <p class="error">{{error}}</p>
    {% endif %}

    {% if result %}
    <div class="output">
        <textarea id="outputText" readonly>{{result}}</textarea>
        <button class="copy-btn" onclick="copyText()">Copy Output</button>
    </div>
    {% endif %}
</div>

<script>
function copyText() {
    const text = document.getElementById("outputText");
    text.select();
    document.execCommand("copy");
}
</script>

</body>
</html>
'''

@app.route("/", methods=["GET", "POST"])
def index():
    if request.method == "POST":
        text = request.form.get("plaintext")
        method = request.form.get("method")

        if not text:
            return render_template_string(HTML_PAGE, error="Text cannot be empty")

        if not method:
            return render_template_string(HTML_PAGE, error="Select encryption method")

        if method == "caesar":
            result = caesar_encrypt(text)

        elif method == "aes":
            key, nonce, ciphertext = aes_encrypt(text)
            result = f"Key: {key}\nNonce: {nonce}\nCipher: {ciphertext}"

        elif method == "base64":
            result = base64_encrypt(text)

        return render_template_string(HTML_PAGE, result=result)

    return render_template_string(HTML_PAGE)

from pyngrok import ngrok

ngrok.set_auth_token(" ") ##Enter your own Ngrok Token here

public_url = ngrok.connect(5000)
print("Your App URL:", public_url)

app.run(port=5000)

Your App URL: NgrokTunnel: "https://ahead-deduct-detail.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [06/May/2026 17:18:26] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/May/2026 17:18:32] "POST / HTTP/1.1" 200 -
